# 01 – Fundamentals: Ev Enerji Verisini Keşfetme

Bu notebook'ta CSV dosyasını açacak, ilk kayıtları inceleyecek, bir gün seçecek ve `Appliances` tüketimini grafikle göstereceğiz. Son olarak günlük toplamı Wh ve kWh cinsinden hesaplayacağız.

**Ön koşul:** `energydata_complete.csv` dosyası proje kökündeki `data/raw/` klasöründe bulunmalı. Bu çalışmada model eğitimi yapılmaz.

## 1. Gerekli kütüphaneler

`pandas` tablo verisiyle, `matplotlib` ise grafiklerle çalışmamızı sağlar.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

# Proje kökünü ve src klasörünü Python'ın arama yoluna ekle.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from energy_forecast.data_analysis.data_loader import load_energy_data

## 2. CSV dosyasını yükle

Hazırladığımız `load_energy_data()` fonksiyonu dosyayı okur ve tarihleri, gerekli sütunları ve 10 dakikalık aralıkları kontrol eder. Dosya bulunamazsa önce `data/raw/` konumunu kontrol et.

In [ ]:
data = load_energy_data()
print("Veri başarıyla yüklendi.")
print(f"Satır sayısı: {len(data):,}")
print(f"Sütun sayısı: {len(data.columns)}")

## 3. İlk satırları incele

`head()` tablonun ilk beş satırını gösterir. `Appliances` sütunu 10 dakikalık ölçüm aralığındaki enerji tüketimini **Wh** cinsinden verir.

In [ ]:
data.head()

Sütun adlarını ve veri türlerini de inceleyelim. `date` tarih, `Appliances` ise sayısal enerji tüketimi değeridir.

In [ ]:
data.info()

## 4. Bir gün seç

Önce veri setinde bulunan günleri listele. Ardından aşağıdaki `selected_day` değerini listeden istediğin bir tarihle değiştir. Tam bir günün 10 dakikalık 144 kaydı bulunur; veri setinin başlangıç ve bitiş günleri kısmi olabilir.

In [ ]:
available_days = sorted(data["date"].dt.date.unique())
print(f"Veri setindeki gün sayısı: {len(available_days)}")
print("İlk 10 gün:", available_days[:10])
print("Son 10 gün:", available_days[-10:])

In [ ]:
# Listeden bir gün seç. İstersen tarihi değiştir.
selected_day = available_days[1]

daily_data = data[data["date"].dt.date == selected_day].copy()
print(f"Seçilen gün: {selected_day}")
print(f"Kayıt sayısı: {len(daily_data)}")

if daily_data.empty:
    raise ValueError("Seçilen gün için veri bulunamadı.")

## 5. 10 dakikalık tüketim grafiği

Yatay eksende ölçüm zamanı, dikey eksende `Appliances` tüketimi yer alır. Her nokta bir 10 dakikalık ölçüm aralığını temsil eder.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily_data["date"], daily_data["Appliances"], marker=".", linewidth=1)
plt.title(f"Appliances – {selected_day}")
plt.xlabel("Saat")
plt.ylabel("Enerji tüketimi (Wh / 10 dakika)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Günlük toplam tüketimi hesapla

Seçilen günün 10 dakikalık Wh değerlerini topluyoruz. Wh değerini kWh'ye çevirmek için 1.000'e böleriz. Gün 144 kayıttan azsa bunun veri setinin kısmi günü olabileceğini unutma.

In [ ]:
daily_total_wh = daily_data["Appliances"].sum()
daily_total_kwh = daily_total_wh / 1000

print(f"Günlük toplam: {daily_total_wh:.2f} Wh")
print(f"Günlük toplam: {daily_total_kwh:.3f} kWh")
print(f"Ölçüm sayısı: {len(daily_data)} / 144")

## 7. Mini alıştırmalar

1. `selected_day` değerini başka bir tarihle değiştir. Grafiğin şekli ve günlük toplam nasıl değişiyor?
2. Seçtiğin günün en yüksek ve en düşük 10 dakikalık `Appliances` tüketimlerini bul. Hangi saatlerde gerçekleştiğini yazdır.
3. Günlük toplamı kWh yerine Wh olarak gösteren satırı bul. Dönüşümde neden 1.000'e böldüğümüzü kendi cümlelerinle açıkla.

**Hatırlatma:** Bu notebook yalnızca veri yükleme, inceleme, görselleştirme ve basit hesaplama içerir. Makine öğrenmesi modeli eğitilmez.